# H2-PRISM Baseline Models Evaluation (Person 2: Faisal)
Compares Logistic Regression, Random Forest, and XGBoost using labeled windows from the selected H2-SimNet scenarios.

## 1- Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

from src.features import extract_window_features
from src.fingerprint import extract_transient_fingerprint
from src.physics import compute_mass_balance_residual
from src.propagation import estimate_propagation_delay

## 2- Load actual scenarios and create labeled windows

Windows are split at the sample level, so results measure performance on these recordings and should not be interpreted as generalization to unseen scenarios.

In [24]:
project_root = Path.cwd()
if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent

data_dir = (project_root / "data" / "raw").resolve()
scenario_files = sorted(data_dir.rglob("*.csv"))
if not scenario_files:
    raise FileNotFoundError(f"No raw scenario CSVs found in {data_dir}")

window_size = 100
required_columns = {
    "timestamp", "scenario_id", "p_upstream", "p_downstream",
    "m_inlet", "m_outlet", "label",
}
scenario_features = []


def load_raw_pipeline_scenario(scenario_path):
    scenario = pd.read_csv(scenario_path)
    relative_path = scenario_path.resolve().relative_to(data_dir)
    scenario_id = "__".join(relative_path.with_suffix("").parts)

    if "timestamp" not in scenario.columns:
        scenario["timestamp"] = np.arange(len(scenario), dtype=float)
    if "scenario_id" not in scenario.columns:
        scenario["scenario_id"] = scenario_id
    if "run_id" not in scenario.columns:
        scenario["run_id"] = relative_path.as_posix()

    if "p_upstream" not in scenario.columns and "PS1" in scenario.columns:
        scenario["p_upstream"] = scenario["PS1"] * 1e6
    if "p_downstream" not in scenario.columns:
        downstream_sensor = "PS12" if "PS12" in scenario.columns else "PS6"
        if downstream_sensor in scenario.columns:
            scenario["p_downstream"] = scenario[downstream_sensor] * 1e6
    if "m_inlet" not in scenario.columns and "MFS1" in scenario.columns:
        scenario["m_inlet"] = scenario["MFS1"]
    if "m_outlet" not in scenario.columns:
        if "anom_gestcent" in {part.lower() for part in relative_path.parts} and "MFS3" in scenario.columns:
            outlet_sensor = "MFS3"
        else:
            outlet_sensor = next(
                (sensor for sensor in ("MFS4", "MFS2", "MFS3") if sensor in scenario.columns),
                None,
            )
        if outlet_sensor is not None:
            scenario["m_outlet"] = scenario[outlet_sensor]

    if "label" not in scenario.columns:
        labels = pd.Series("NORMAL", index=scenario.index, dtype="object")
        if "label_anomaly" in scenario.columns:
            labels.loc[scenario["label_anomaly"] == 1] = "ANOMALY"
        for column in ("leak_G3", "leak_G4", "leak_G5", "leak_G6"):
            if column in scenario.columns:
                labels.loc[scenario[column] == 1] = "LEAK"
        if "CompressorFault" in scenario.columns:
            labels.loc[scenario["CompressorFault"] == 1] = "COMPRESSOR_FAULT"
        if "CompressorDelay" in scenario.columns:
            labels.loc[scenario["CompressorDelay"] == 1] = "COMPRESSOR_DELAY"
        scenario["label"] = labels

    return scenario


for scenario_path in scenario_files:
    scenario = load_raw_pipeline_scenario(scenario_path)
    missing_columns = required_columns.difference(scenario.columns)
    if missing_columns:
        raise ValueError(
            f"{scenario_path.name} is missing columns: {sorted(missing_columns)}"
        )

    for start in range(0, len(scenario) - window_size + 1, window_size):
        window = scenario.iloc[start : start + window_size]
        window_labels = window["label"].dropna().astype(str)
        if window_labels.empty:
            continue

        features = extract_window_features(window)
        features["scenario_id"] = scenario["scenario_id"].iloc[0]
        features["window_start"] = start
        features["label"] = window_labels.value_counts().idxmax()
        scenario_features.append(features)

feature_table = pd.DataFrame(scenario_features)
if feature_table.empty:
    raise ValueError("No labeled windows were created from the raw scenarios.")

print(f"Scenarios loaded: {len(scenario_files)}")
print(f"Labeled windows: {len(feature_table)}")
print(feature_table["label"].value_counts().sort_index())

Scenarios loaded: 57
Labeled windows: 11400
label
COMPRESSOR_DELAY      30
COMPRESSOR_FAULT    2400
LEAK                7191
NORMAL              1779
Name: count, dtype: int64


In [25]:
excluded_columns = {"label", "scenario_id", "window_start"}

feature_names = [
    column
    for column in feature_table.columns
    if column not in excluded_columns
]

feature_table[feature_names] = (
    feature_table[feature_names]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

# Use GroupShuffleSplit to prevent data leakage by keeping scenarios intact
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_indices, test_indices = next(
    gss.split(feature_table, groups=feature_table["scenario_id"])
)

train_table = feature_table.loc[train_indices].copy()
test_table = feature_table.loc[test_indices].copy()

# --- THE FIX: Filter out unseen classes from the test set ---
known_labels = train_table["label"].unique()
test_table = test_table[test_table["label"].isin(known_labels)].copy()
# ------------------------------------------------------------

X_train = train_table[feature_names]
X_test = test_table[feature_names]

y_train = train_table["label"]
y_test = test_table["label"]

## 3- Train and compare baseline models

In [26]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

models["XGBoost"] = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

evaluation_rows = []
for model_name, model in models.items():
    if model_name == "XGBoost":
        model.fit(X_train, y_train_encoded)
        predictions_encoded = model.predict(X_test)
        predictions = label_encoder.inverse_transform(
            predictions_encoded.astype(int)
        )
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

    balanced_accuracy = balanced_accuracy_score(y_test, predictions)
    evaluation_rows.append(
        {"model": model_name, "balanced_accuracy": balanced_accuracy}
    )

    print(f"\n{model_name} | balanced accuracy: {balanced_accuracy:.3f}")
    print(classification_report(y_test, predictions, zero_division=0))

evaluation_table = pd.DataFrame(evaluation_rows).sort_values(
    "balanced_accuracy",
    ascending=False,
    ignore_index=True,
)
display(evaluation_table)

C:\Users\faisa\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



Logistic Regression | balanced accuracy: 0.680
                  precision    recall  f1-score   support

COMPRESSOR_DELAY       0.00      0.00      0.00         0
COMPRESSOR_FAULT       1.00      0.35      0.51       600
            LEAK       1.00      0.69      0.82      2060
          NORMAL       0.35      1.00      0.52       340

        accuracy                           0.66      3000
       macro avg       0.59      0.51      0.46      3000
    weighted avg       0.93      0.66      0.72      3000


Random Forest | balanced accuracy: 0.954
                  precision    recall  f1-score   support

COMPRESSOR_FAULT       1.00      1.00      1.00       600
            LEAK       1.00      0.86      0.93      2060
          NORMAL       0.54      1.00      0.70       340

        accuracy                           0.90      3000
       macro avg       0.85      0.95      0.88      3000
    weighted avg       0.95      0.90      0.92      3000


XGBoost | balanced accuracy: 0.94

,model,balanced_accuracy
0,Random Forest,0.953722
1,XGBoost,0.949172
2,Logistic Regression,0.680442
